## Importaciones de Librerías

In [57]:
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

from lightgbm import LGBMClassifier

import optuna

import joblib

import requests

from dotenv import load_dotenv
import os

## Carga de Datos

In [58]:
load_dotenv()
URL_BACKEND_INTERNSHIPS = os.getenv('URL_BACKEND_INTERNSHIPS')

In [59]:
def get_internships():
    try:
        response = requests.get(URL_BACKEND_INTERNSHIPS)
        if response.status_code == 200:
            return response.json()
        else:
            print(f'Error al obtener los datos: {response.status_code} - {response.text}')
            return None
    except Exception as e:
        print(f'Error con la conexión a la API de las prácticas: {e}')
        return None

In [60]:
df = pd.DataFrame(get_internships())
print("Datos cargados exitosamente")

Datos cargados exitosamente


## Preparación Varibales x, y

In [61]:
# Definimos solo las variables que demostraron verdadero valor predictivo
variables_seleccionadas = [
    'interview_score', 'skills_score', 'communication_score', 'coding_test_score',
    'projects_count', 'soft_skills_score', 'certifications_count', 'resume_score',
    'internships_done', 'extracurricular', 'consistency_score', 'placement_training',
    'cgpa', 'github_score'
]

X = df[variables_seleccionadas]
y = df['selected']

## Separación varibales en entreno y test

In [62]:
X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.8, random_state=42, stratify=y)

## Búsuqueda de Mejores Hiperparametros con Optuna

In [63]:
def objective_lightgbm(trial):
    params = {
        'boosting_type': trial.suggest_categorical('boosting_type', ['rf', 'gbdt']),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'max_depth': trial.suggest_int('max_depth', 5, 20),
        'num_leaves': 31,
        'bagging_freq': 1,
        'bagging_fraction': 0.8,
        'feature_fraction': 0.8,
        'objective': 'binary',
        'random_state': 42
    }

    model = LGBMClassifier(**params)
    return cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc').mean()
study_lgb = optuna.create_study(direction='maximize')
study_lgb.optimize(objective_lightgbm, n_trials=25)
print('LightGBM - Mejores Hiperparámetros: ', study_lgb.best_params)

[I 2026-06-25 00:22:36,390] A new study created in memory with name: no-name-750724f4-6edb-49dc-8db2-6699ad0aafb2


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Info] Number of positive: 4719, number of negative: 1681
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002173 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 363
[LightGBM] [Info] Number of data point

[I 2026-06-25 00:22:37,859] Trial 0 finished with value: 0.545102322460828 and parameters: {'boosting_type': 'gbdt', 'learning_rate': 0.19466750567331956, 'max_depth': 9}. Best is trial 0 with value: 0.545102322460828.


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Info] Number of positive: 4719, number of negative: 1681
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000716 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 363
[LightGBM] [Info] Number of data point

[I 2026-06-25 00:22:38,945] Trial 1 finished with value: 0.5688258655241757 and parameters: {'boosting_type': 'rf', 'learning_rate': 0.012106021188331293, 'max_depth': 12}. Best is trial 1 with value: 0.5688258655241757.


...

[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 

[I 2026-06-25 00:22:54,018] Trial 23 finished with value: 0.5737488425313618 and parameters: {'boosting_type': 'gbdt', 'learning_rate': 0.021391493250911938, 'max_depth': 15}. Best is trial 7 with value: 0.5747583948906738.


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 

[I 2026-06-25 00:22:54,677] Trial 24 finished with value: 0.5733768528623987 and parameters: {'boosting_type': 'gbdt', 'learning_rate': 0.010529564336513652, 'max_depth': 15}. Best is trial 7 with value: 0.5747583948906738.


[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.8, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.8
[LightGBM] [Warning] bagging_fraction is set=0.8, subsample=1.0 will be ignored. Current value: bagging_fraction=0.8
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 

## Pipeline

In [64]:
# Número de negativos / positivos en tu set de entrenamiento
num_negativos = (y_train == 0).sum()
num_positivos = (y_train == 1).sum()
ratio_peso = num_negativos / num_positivos

pipeline = Pipeline(
    steps = [
        ('model', LGBMClassifier(**study_lgb.best_params, scale_pos_weight=ratio_peso, eval_metric='logloss', random_state=42))
    ]
)

In [65]:
print("Entrenando el modelo final con los mejores parámetros...")
pipeline.fit(X_train, y_train)

Entrenando el modelo final con los mejores parámetros...
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Info] Number of positive: 5899, number of negative: 2101
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001325 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 363
[LightGBM] [Info] Number of data points in the train set: 8000, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.737375 -> initscore=1.032369
[LightGBM] [Info] Start training from score 1.032369


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[<U20](14,)","['interview_score','skills_score','communication_score',..., 'placement_training','cgpa','github_score']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,14
,max_depth,16
,learning_rate,0.04157262733798423
,random_state,42


In [66]:
# Calcular predicciones para ambos conjuntos
y_train_pred_proba = pipeline.predict_proba(X_train)[:, 1]
y_test_pred_proba = pipeline.predict_proba(X_test)[:, 1]

# Calcular el ROC-AUC para ambos
auc_train = roc_auc_score(y_train, y_train_pred_proba)
auc_test = roc_auc_score(y_test, y_test_pred_proba)

print(f"ROC-AUC Entrenamiento: {auc_train:.4f}")
print(f"ROC-AUC Prueba (Test):  {auc_test:.4f}")

# Calcular la brecha (gap)
brecha = auc_train - auc_test
print(f"Brecha de Overfitting:   {brecha:.4f}")

[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] Unknown parameter: eval_metric
ROC-AUC Entrenamiento: 0.8548
ROC-AUC Prueba (Test):  0.5547
Brecha de Overfitting:   0.3001


In [67]:
print("\nEvaluando el modelo en el conjunto de prueba (X_test)...")
y_pred = pipeline.predict(X_test)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]


Evaluando el modelo en el conjunto de prueba (X_test)...
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] Unknown parameter: eval_metric


In [68]:
print("\n=== Reporte de Clasificación ===")
print(classification_report(y_test, y_pred))

print("=== Matriz de Confusión ===")
print(confusion_matrix(y_test, y_pred))

print(f"\nROC-AUC Score en Test: {roc_auc_score(y_test, y_pred_proba):.4f}")


=== Reporte de Clasificación ===
              precision    recall  f1-score   support

           0       0.31      0.46      0.37       525
           1       0.77      0.63      0.69      1475

    accuracy                           0.58      2000
   macro avg       0.54      0.54      0.53      2000
weighted avg       0.64      0.58      0.61      2000

=== Matriz de Confusión ===
[[241 284]
 [547 928]]

ROC-AUC Score en Test: 0.5547


## Guardado del modelo

In [69]:
joblib.dump(pipeline, '../models/lgb_model.pkl')

['../models/lgb_model.pkl']